In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]


tau = length(Istar_obs)

model_tag_sym = :powerlaw

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 1

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [powerlaw_model] Fitting chain 1 (tau=34)
[ Info: [powerlaw] iter 1000/1000000 elapsed=3.9s, rate=0.158, mean=[1.891, 0.00099, 1.656, 0.489], std=[0.3122, 0.000386, 0.0630, 0.0853] [ADAPT]
[ Info: [powerlaw] iter 2000/1000000 elapsed=7.1s, rate=0.127, mean=[2.046, 0.00087, 1.737, 0.500], std=[0.2676, 0.000302, 0.1056, 0.0619] [ADAPT]
[ Info: [powerlaw] iter 3000/1000000 elapsed=9.3s, rate=0.112, mean=[2.145, 0.00082, 1.785, 0.499], std=[0.2539, 0.000260, 0.1086, 0.0517] [ADAPT]
[ Info: [powerlaw] iter 4000/1000000 elapsed=11.5s, rate=0.102, mean=[2.193, 0.00079, 1.789, 0.508], std=[0.2342, 0.000236, 0.0954, 0.0473] [ADAPT]
[ Info: [powerlaw] iter 5000/1000000 elapsed=13.7s, rate=0.098, mean=[2.218, 0.00078, 1.817, 0.511], std=[0.2166, 0.000218, 0.1016, 0.0464] [ADAPT]
[ Info: [powerlaw] iter 6000/1000000 elapsed=15.9s, rate=0.097, mean=[2.241, 0.00077, 1.868, 0.515], std=[0.2057, 0.000205, 0.1406, 0.0442] [ADAPT]
[ Info: [powerlaw] iter 7000/1000000 elapsed=18.1s, rate=0.098, m